<a href="https://colab.research.google.com/github/Hania-Emaan/urdu-ocr-codesaviours-si26-Hania-/blob/main/SI26_Week3_Hania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import sentencepiece
print(sentencepiece.__version__)

0.2.2


In [6]:
!pip install -q transformers==4.44.2 tokenizers==0.19.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [4]:
from transformers import RobertaTokenizer, ViTImageProcessor

tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-printed')
image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-printed')

print('Loaded successfully!')

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

Loaded successfully!


In [5]:
import torch, os
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, img_root, tokenizer, image_processor):
        self.data = pd.read_csv(csv_path, encoding='utf-8')
        self.img_root = img_root
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(os.path.join(self.img_root, row['image'])).convert('RGB')
        encoding = self.image_processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()
        labels = self.tokenizer(row['text'], padding='max_length', max_length=128, truncation=True).input_ids
        labels = torch.tensor(labels)
        return {'pixel_values': pixel_values, 'labels': labels}

In [6]:
raw_base = '/content/drive/MyDrive/urdu-ocr-si26/data/raw'
csv_path = '/content/drive/MyDrive/urdu-ocr-si26/data/labels.csv'

dataset = UrduOCRDataset(csv_path, raw_base, tokenizer, image_processor)
sample = dataset[0]
print('pixel_values shape:', sample['pixel_values'].shape)
print('labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

Dataset loaded: 209 samples
pixel_values shape: torch.Size([3, 384, 384])
labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 167
Testing samples: 42


**Note on implementation:** The handout's original code uses `TrOCRProcessor.from_pretrained(...)`
to load both the image processor and tokenizer together. However, this caused a persistent
`ValueError` due to a version incompatibility between the installed `transformers` library and
the TrOCR tokenizer's fast-tokenizer conversion path (a known issue unrelated to the dataset
or code logic).

As a workaround, I loaded the tokenizer (`RobertaTokenizer`) and image processor
(`ViTImageProcessor`) separately — these are the same two components `TrOCRProcessor` combines
internally, so functionality is identical. The Dataset class below uses both directly.